In [1]:
from sklearn.datasets import load_iris

x, y = load_iris(return_X_y=True)

print("x shape:", x.shape)
print("y shape:", y.shape)

x shape: (150, 4)
y shape: (150,)


In [2]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [3]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(
    score_func=f_classif,
    k=2
)

x_train_selected = selector.fit_transform(
    x_train,
    y_train
)

x_test_selected = selector.transform(
    x_test
)

print("Original shape:", x_train.shape)
print("Selected shape:", x_train_selected.shape)

Original shape: (120, 4)
Selected shape: (120, 2)


In [4]:
print(selector.get_support())

[False False  True  True]


In [7]:
print(selector.scores_)

[100.96592271  36.03057712 948.89038101 803.2140881 ]


In [8]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier()

model.fit(
    x_train_selected,
    y_train
)

accuracy = model.score(
    x_test_selected,
    y_test
)

print("Test Accuracy:", accuracy)

Test Accuracy: 0.9666666666666667


# 05. Feature Selection

## 1. What is Feature Selection?

Feature Selection is the process of **selecting the most useful features** from the available features.

```text
All Features
     ↓
Evaluate Feature Importance
     ↓
Select Useful Features
     ↓
Train Model
```

Feature Selection **removes features**; it does not create new ones.

---

## 2. Why Feature Selection?

### Reduce unnecessary features

```text
100 Features
     ↓
Select 20 useful Features
```

### Reduce computation

Fewer features can reduce the amount of computation required during training and prediction.

### Reduce overfitting

Irrelevant or noisy features can sometimes make a model learn unnecessary patterns.

### Improve interpretability

A model using fewer relevant features can be easier to understand.

---

# 3. Feature Selection vs PCA

This is an important distinction.

### Feature Selection

Keeps the **original features**.

```text
Original:

Age
Income
Height
Weight
Salary

        ↓

Selected:

Age
Income
Salary
```

No new features are created.

### PCA

Creates **new components** from the original features.

```text
Original Features
       ↓
      PCA
       ↓
PC1
PC2
```

Therefore:

```text
Feature Selection
→ Select existing features

PCA
→ Create new transformed components
```

---

# 4. SelectKBest

`SelectKBest` selects the **K best features** according to a scoring function.

```python
from sklearn.feature_selection import SelectKBest, f_classif
```

Example:

```python
selector = SelectKBest(
    score_func=f_classif,
    k=2
)
```

Here:

```text
score_func=f_classif
→ Method used to score the features

k=2
→ Select the best 2 features
```

---

# 5. Fit Feature Selector

```python
x_train_selected = selector.fit_transform(
    x_train,
    y_train
)

x_test_selected = selector.transform(
    x_test
)
```

### Training data

```text
fit
→ Learn which features are useful

transform
→ Keep only selected features
```

Therefore:

```text
x_train
   ↓
fit_transform()
   ↓
x_train_selected
```

### Test data

```text
x_test
   ↓
transform()
   ↓
x_test_selected
```

We don't call `fit()` on the test data because the test set should not influence feature selection.

---

# 6. `get_support()`

```python
print(selector.get_support())
```

Our result:

```text
[False False True True]
```

This tells us which features were selected.

For Iris:

```text
Feature 0 → Sepal Length → ❌
Feature 1 → Sepal Width  → ❌
Feature 2 → Petal Length → ✅
Feature 3 → Petal Width  → ✅
```

Therefore:

```text
Selected Features:
→ Petal Length
→ Petal Width
```

---

# 7. Feature Scores

```python
print(selector.scores_)
```

Our result:

```text
[100.96592271  36.03057712  948.89038101  803.2140881]
```

Meaning:

```text
Sepal Length → 100.97
Sepal Width  → 36.03
Petal Length → 948.89
Petal Width  → 803.21
```

Higher score means a stronger relationship with the target classes according to `f_classif`.

Therefore:

```text
Petal Length → 948.89 🥇
Petal Width  → 803.21 🥈
Sepal Length → 100.97
Sepal Width  → 36.03
```

So `SelectKBest(k=2)` selected the two highest-scoring features.

---

# 8. Train Model Using Selected Features

```python
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier()

model.fit(
    x_train_selected,
    y_train
)

accuracy = model.score(
    x_test_selected,
    y_test
)

print("Test Accuracy:", accuracy)
```

Our result:

```text
Test Accuracy = 0.9667
```

Therefore:

```text
96.67%
```

---

# 9. Effect of Feature Selection

Using all 4 features:

```text
KNN Test Accuracy
→ 100%
```

Using only the selected 2 features:

```text
KNN Test Accuracy
→ 96.67%
```

So in our particular experiment:

```text
4 Features
   ↓
100% Accuracy

2 Selected Features
   ↓
96.67% Accuracy
```

We lost a small amount of accuracy, but reduced the number of features from **4 to 2**.

---

# 10. Complete Workflow

```text
Dataset
   ↓
Train / Test Split
   ↓
Feature Selection
   ↓
Score Features
   ↓
Select Top K Features
   ↓
Transform Training Data
   ↓
Transform Test Data
   ↓
Train Model
   ↓
Evaluate Model
```

---

## ⭐ Key Points

```text
Feature Selection
→ Select the most useful existing features.
```

```text
SelectKBest
→ Selects the K highest-scoring features.
```

```text
f_classif
→ Uses ANOVA F-test to score features for classification.
```

```text
get_support()
→ Shows which features were selected.
```

```text
fit_transform()
→ Learn selection + transform training data.
```

```text
transform()
→ Apply the already-learned selection to test data.
```

```text
Feature Selection
→ Removes features

PCA
→ Creates new components
```

### One-line summary

> **Feature Selection reduces the number of input features by keeping the features that provide the most useful information for the prediction task.**
